In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts import ChatPromptTemplate
from config import OPENAI_API_KEY
from process_senses import process_senses_with_chain, default_build_senses_block, parse_model_output
from writers import CustomWebAnnoTSVWriter, InceptionWebAnnoTSVWriter
from config import SENSE_REPO, OUTPUT_DIR
import pandas as pd

# Set the model origin for traceability
ORIGIN_LLM = "ChatGPT"

In [2]:
# 1) Instantiate the LLM
# gpt-5 does not support temperature=0 (API returned unsupported_value). Use default temperature.
llm = ChatOpenAI(
    model="gpt-5",
    openai_api_key=OPENAI_API_KEY,
 )
# Update origin label to reflect exact model
ORIGIN_LLM = "gpt-5"

# 2) Define the chat prompt template using the latest API style
# Reverted to original prompt formatting with double braces for the JSON example
system_message = """
Vi ste ekspert za leksiku i semantiku. Na osnovu konteksta rečenice i liste validnih značenja ciljne reči,
vaš zadatak je da identifikujete ono značenje koje se najpreciznije koristi u datom kontekstu.

Odgovor mora biti u strogo definisanom JSON formatu:

{{
  "sense_id": "<jedan od ponuđenih ID-jeva ili 'NEW_SENSE'>",
  "explanation": "<kratko i jasno obrazloženje u jednoj ili dve rečenice zašto je to značenje primenjivo. Ako se koristi 'NEW_SENSE', objasnite zašto nijedno ponuđeno značenje ne odgovara.>"
}}

Ne dodajete nikakav dodatni tekst van JSON strukture.
Koristite 'NEW_SENSE' samo ako nijedno značenje nije čak ni približno tačno u kontekstu.
Sense_id mora biti identičan jednom od ponuđenih ID-jeva ili tačno 'NEW_SENSE'; nikada ne smete izmišljati druge ID-ove.
"""

# Removed the escaped forward slash to avoid invalid escape sequence warning
user_prompt = """
Kontekst rečenice (ciljna reč je označena HTML tagom <b>...</b>):
"{sentence}"

Ciljna reč: "{word}"

Lista mogućih značenja:
{senses_block}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    ("user", user_prompt)
])

parser = StrOutputParser()

chain = prompt | llm | parser

In [6]:
# 3) Load sense repository and sentences
from data_loader import load_sense_repo
senses_df = load_sense_repo()

from webanno_spacy_converter.parsers.tsv_parser_v3 import WebAnnoLEXISParser
from config import ANNOTATION_CHUNKS, DATA_DIR

# Human-readable chunk selection
chunks = [(b, e, (DATA_DIR / fname)) for (b, e, fname) in ANNOTATION_CHUNKS]
print("Available chunks (index, begin, end, file):",
      [(i, b, e, p.name) for i, (b, e, p) in enumerate(chunks)])

# Choose which chunk to process here (0-based)
chunk_idx = 0
chunk_begin, chunk_end, selected_path = chunks[chunk_idx]
print(f"Using chunk #{chunk_idx}: {selected_path.name} -> ({chunk_begin}, {chunk_end})")

parser = WebAnnoLEXISParser(selected_path)
senteces = parser.parse()

# Testing flag: when True, take only a small slice; when False, keep full chunk
# By default, use chunk boundaries for filenames
begin, end = chunk_begin, chunk_end

test = True
if test:
    # Use a predictable small slice for quick dry runs
    tb, te = 0, 20
    senteces = senteces[tb:te]
    # Human-facing indices in filenames (1-based start, inclusive end)
    begin, end = tb + 1, te

Available chunks (index, begin, end, file): [(0, 1, 500, 'sr-elexis-WSD_0001_0500.tsv'), (1, 501, 1000, 'sr-elexis-WSD_0501_1000.tsv'), (2, 1001, 1500, 'sr-elexis-WSD_1001_1500.tsv'), (3, 1501, 2000, 'sr-elexis-WSD_1501_2000.tsv'), (4, 2001, 2024, 'sr-elexis-WSD_2001_2024.tsv')]
Using chunk #0: sr-elexis-WSD_0001_0500.tsv -> (1, 500)


In [7]:
# 4) Annotate senses using the shared utility
senteces = process_senses_with_chain(
    senteces,
    senses_df,
    chain,
    ORIGIN_LLM,
    build_senses_block=default_build_senses_block,
    parse_json_response_clean=parse_model_output
)

Processed 20/20 sentences.

In [ ]:
# 5) Save outputs (Second round: add _IIIround suffix)
ROUND_SUFFIX = "_test"
writer = CustomWebAnnoTSVWriter(senteces)
# Always include begin/end in filenames to reflect chunk or test slice
writer.save(OUTPUT_DIR / f"LexiSense_{begin:04d}_{end:04d}_{ORIGIN_LLM}{ROUND_SUFFIX}.tsv")

incept_writer = InceptionWebAnnoTSVWriter(senteces)
incept_writer.save(OUTPUT_DIR / f"LexiSense_Inception_{begin:04d}_{end:04d}_{ORIGIN_LLM}{ROUND_SUFFIX}.tsv")

: 

# ChatGPT_sense.ipynb

This notebook demonstrates sense annotation for Serbian using the ChatGPT model and a shared, reusable pipeline. All model-specific and shared logic is modularized for maintainability.